In [ ]:
import os

# List the current directory
#print(os.listdir(.))

In [ ]:
# Ask the user to select a digit in the range 0 to 8
k = int(input("Please select a digit in the range 0 to 8: "))

# Check if the digit is within the valid range
if 0 <= k <= 8:
    print(f"will be running in k{k}")
else:
    print("Invalid selection. Please select a digit between 0 and 8.")
    exit(1)

from pathlib import Path

wd = Path(f"/home/rfriedma/src/k{k}/ceph/build")
os.chdir(wd)
%env CEPH_JTEST_ROOT=/home/rfriedma/src/k{k}/ceph/build
!echo $CEPH_JTEST_ROOT > /tmp/jpath


In [ ]:
!ls -l
!bash -c MGR=0 ls -l


In [ ]:
%%bash

function get_pg() {
    local poolname=$1
    local objectname=$2
    bin/ceph --format json osd map $poolname $objectname 2>/dev/null | jq -r '.pgid'
}

function get_primary() {
    local poolname=$1
    local objectname=$2

    bin/ceph --format json osd map $poolname $objectname 2>/dev/null | \
        jq '.acting_primary'
}



scrtch=to_"`date +%d_%H%M`"
echo $scrtch

MDS=0 MGR=1 OSD=3 MON=1 ../src/vstart.sh -n  --without-dashboard --msgr2 -X --memstore -o "memstore_device_bytes=68435456" -o "osd_op_queue=wpq"
sleep 2
bin/ceph -s

#bin/ceph tell osd.* config set debug_osd 20/20

bin/ceph config set global osd_pool_default_pg_autoscale_mode off
sleep 2

# disable rescheduling of the queue due to 'no-scrub' flags
bin/ceph tell osd.* config set osd_scrub_backoff_ratio 0.9999

# initial set of global scrub scheduling parameters
bin/ceph tell osd.* config set osd_scrub_interval_randomize_ratio 0.1
bin/ceph tell osd.* config set osd_deep_scrub_randomize_ratio 0
# bin/ceph tell osd.* config set osd_scrub_min_interval 10
# bin/ceph tell osd.* config set osd_scrub_max_interval 2000
# bin/ceph tell osd.* config set osd_deep_scrub_interval 600


#PL1 is of size 3

bin/ceph osd pool create pl1 4 4
sleep 1
bin/ceph osd pool autoscale-status
pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo "PL1 #: $pl1_num"
bin/ceph osd pool set pl1 size 3
bin/ceph osd pool set pl1 min_size 3
bin/ceph osd pool set pl1 pg_autoscale_mode off
bin/ceph osd pool stats
bin/ceph osd pool set pl1 noscrub 0
bin/ceph osd pool set pl1 nodeep-scrub 0
sleep 2

#bin/rados bench -p pl1 -t 1 1 write -b 4096 --max-objects 8  --no-cleanup; 
#bin/rados bench -p pl1 1 write -b 4096 --max-objects 128 --show-time --no-cleanup --run-name eeeee


In [ ]:
%%bash

set -x

function get_pg() {
    local poolname=$1
    local objname=$2
    bin/ceph --format json osd map $poolname $objname 2>/dev/null | jq -r '.pgid'
}

function create_obj() {
    local poolname=$1
    local objname=$2
    local data_1k_blocks=$3

    # create an object in the specified pool
    dd if=/dev/urandom bs=1024 count=$data_1k_blocks | bin/rados -p "$poolname" put "$objname" - || return 1

    # add some omap entries:
    bin/rados --pool "$poolname" setomapheader "$objname" "hdr-$objname" || return 1
    for i in $(seq 1 5); do
        # add some random omap entries
        local key="omap_key_$i"
        dd if=/dev/urandom bs=1024 count=16 | bin/rados --pool "$poolname" setomapval "$objname" "$key" || return 1
    done
    bin/rados --pool "$poolname" setomapval "$objname" "omap_key_final" "final_value" || return 1
}

function dump_io() {
    local poolname="$1"
    local msg="$2"
    #local osd_count=$(bin/ceph osd tree | grep -c 'osd\.[0-9]\+')
    local osd_count=3
    local dist1=$(( ( RANDOM % 500 )  + 1000 ))
    local basefn="/tmp/osdio_${dist1}_${msg}_"
    #echo "# OSDs: $osd_count <<$msg>> <<$dist>>"
    echo "-> $basefn"
    for i in $(seq 0 $((osd_count-1))); do
        echo "$i: into: $basefn$i"
        bin/ceph tell osd.$i counter dump >> "$basefn$i" 2>/dev/null
        bin/ceph tell osd.$i counter dump | grep -E '(scrub_pri)|(scrub_rep)|(scrubs_)|(scrub_)'
    done
}



# add objects with both data and omap entries
create_obj pl1 obj1 10

sleep 3
bin/ceph tell osd.* config set debug_osd 20/20

# the set of PGs
bin/ceph pg dump
bin/ceph pg dump pgs_brief
bin/ceph pg dump pgs_brief -f=json-pretty
dump_io pl1 "initial" # dump IO counters for all OSDs

# find a relevant PG to scrub
o1pg=$(get_pg pl1 obj1)
echo "o1pg: $o1pg"
bin/ceph tell ${o1pg} deep-scrub
sleep 4
dump_io pl1 "deep-scrub"
ls -ltr /tmp | tail -6




In [ ]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
jp=$CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    %env CEPH_JTEST_ROOT=$jp
fi
cd $jp


In [ ]:
%%bash

file_path="/tmp/jpath"
if [ -f "$file_path" ]; then
        file_contents=$(cat "$file_path")
        echo "File contents read into variable."
else
        echo "File does not exist."
fi

In [ ]:
%%bash

# find the PG for the object we created
pgid=$(get_pg pl1 obj1)
bin/ceph tell $pgid scrub
sleep 10
dump_io pl1 "after scrub of obj1"



In [ ]:
%%bash

fnm=to_"`date +%d_%H%M`"

bin/ceph tell osd.1 counter dump > /tmp/${fnm}_a
head -20 /tmp/${fnm}_a

echo "=============="

bin/ceph tell osd.1 perf schema > /tmp/${fnm}_b
wc -l /tmp/${fnm}_b
head -20 /tmp/${fnm}_b

echo "=============="

bin/ceph tell osd.1 counter schema > /tmp/${fnm}_c
wc -l /tmp/${fnm}_c
head -20 /tmp/${fnm}_c

# "perf dump" us the old form (only unlabeled?)


#bin/ceph tell osd.1 counter dump | jq 'with_entries(select(.key | startswith("osd_scrub")))' > /tmp/ans2


The rest is irrelevant to the I/O Counters task

In [ ]:
%%bash

#cd $CEPH_JTEST_ROOT

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00_b4params.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01_b4params.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02_b4params.json


# set the scheduling parameters

bin/ceph osd pool set pl1 scrub_min_interval 1
bin/ceph osd pool set pl1 scrub_max_interval 2
bin/ceph osd pool set pl1 deep_scrub_interval 1000

#bin/ceph osd pool set pl2 scrub_min_interval 2
#bin/ceph osd pool set pl2 scrub_max_interval 4
#bin/ceph osd pool set pl2 deep_scrub_interval 10

sleep 3

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02.json
#bin/ceph tell osd.3 dump_scrubs --format=json-pretty > /tmp/ds_02.json





In [ ]:
%%bash

#%cd $CEPH_JTEST_ROOT

# common scrub configs
bin/ceph tell osd.* config set osd_blocked_scrub_grace_period 20
bin/ceph tell osd.* config set osd_stats_update_period_scrubbing 2
bin/ceph tell osd.* config set osd_stats_update_period_not_scrubbing 3
#bin/ceph tell osd.* config set osd_scrub_backoff_ratio 0.9999
#bin/ceph tell osd.* config set osd_scrub_interval_randomize_ratio 0.1
#bin/ceph tell osd.* config set osd_deep_scrub_randomize_ratio 0

#bin/ceph tell mgr.$(bin/ceph mgr services | jq -r .mgr) config set mgr_stats_period 2


In [ ]:
%%bash

# list the scrub queue
scrtch=to_"`date +'%H%M%S'`"
echo $scrtch
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json


In [ ]:
%%bash
# set scrub parameters to guarantee slow scrub
bin/ceph tell osd.* config set osd_scrub_sleep "3.0"
bin/ceph tell osd.* config set osd_max_scrubs 1
bin/ceph tell osd.* config set osd_scrub_chunk_max 5
bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 5


In [ ]:
%%bash

# set higher urgency to one of the PGs
bin/ceph tell $pl1_num.7 scrub
bin/ceph tell $pl1_num.6 schedule-deep-scrub
sleep 1
bin/ceph pg dump pgs


In [ ]:
%%bash

scrtch=to_"`date +'%H%M%S'`"
echo $scrtch

# list the scrub queue
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty


In [ ]:
%%bash

# the idea now is to build on the previous attempt, and:
# 1 - create a dictionary of refs to per-pg dictionary of the data (or - if too complex - the data as a list)
# 2 - return multiple objects:
# 1) a full dict as above
# 2) PGs to primary
# 3) PGs to acting set
# 4) pool to PGs
# 5) PG to pool

function build_pg_dicts {
  local dir=$1
  local -n pg_primary_dict=$2
  local -n pg_acting_dict=$3
  local -n pg_pool_dict=$4
  local infile=$5

  local extr_dbg=2 # note: 3 and above leave some temp files around

  #turn off '-x' (but remember previous state)
  local saved_echo_flag=${-//[^x]/}
  set +x

  # if the infile name is '-', fetch the dump directly from the ceph cluster
  if [[ $infile == "-" ]]; then
    local -r ceph_cmd="bin/ceph pg dump pgs_brief -f=json-pretty"
    local -r ceph_cmd_out=$(eval $ceph_cmd)
    local -r ceph_cmd_rc=$?
    if [[ $ceph_cmd_rc -ne 0 ]]; then
      echo "Error: the command '$ceph_cmd' failed with return code $ceph_cmd_rc"
      #return $ceph_cmd_rc
    fi
    (( extr_dbg >= 3 )) && echo "$ceph_cmd_out" > /tmp/e2
    l0=`echo "$ceph_cmd_out" | jq '[.pg_stats | group_by(.pg_stats)[0] | map({pgid: .pgid, pool: (.pgid | split(".")[0]), acting: .acting, acting_primary: .acting_primary})] | .[]' `
  else
    l0=`jq '[.pg_stats | group_by(.pg_stats)[0] | map({pgid: .pgid, pool: (.pgid | split(".")[0]), acting: .acting, acting_primary: .acting_primary})] | .[]' $infile `
  fi
  (( extr_dbg >= 2 )) && echo "L0: $l0"

  mapfile -t l1 < <(echo "$l0" | jq -c '.[]')
  (( extr_dbg >= 2 )) && echo "L1: ${#l1[@]}"

  for item in "${l1[@]}"; do
    pgid=$(echo "$item" | jq -r '.pgid')
    acting=$(echo "$item" | jq -r '.acting | @sh')
    pg_acting_dict["$pgid"]=$acting
    acting_primary=$(echo "$item" | jq -r '.acting_primary')
    pg_primary_dict["$pgid"]=$acting_primary
    pool=$(echo "$item" | jq -r '.pool')
    pg_pool_dict["$pgid"]=$pool
    #pool_dict["$pgid"]="acting=($acting) acting_primary=$acting_primary pool=$pool"
  done

  if [[ -n "$saved_echo_flag" ]]; then set -x; fi
}

# declare -A pg_pr
# declare -A pg_ac
# declare -A pg_po
# build_pg_dicts . pg_pr pg_ac pg_po "-"
# 
# echo "PGs to primary:"
# for pg in "${!pg_pr[@]}"; do
#   echo "Got: $pg: ${pg_pr[$pg]} ( ${pg_ac[$pg]} ) ${pg_po[$pg]}"
# done



# a function that counts the number of common active-set elements between two PGs
# 1 - the first PG
# 2 - the second PG
# 3 - the dictionary of active sets
function count_common_active {
  local pg1=$1
  local pg2=$2
  local -n pg_acting_dict=$3
  local -n res=$4

  local -a a1=(${pg_acting_dict[$pg1]})
  local -a a2=(${pg_acting_dict[$pg2]})

  local -i cnt=0
  for i in "${a1[@]}"; do
    for j in "${a2[@]}"; do
      if [[ $i -eq $j ]]; then
        cnt=$((cnt+1))
      fi
    done
  done

  res=$cnt
}

# a function that returns an array of the common active-set elements between two PGs
# 1 - the first PG
# 2 - the second PG
# 3 - the dictionary of active sets
function get_common_active {
  local pg1=$1
  local pg2=$2
  local -n actng=$3
  local -n res=$4

  local -a a1=(${actng[$pg1]})
  local -a a2=(${actng[$pg2]})

  local -a common=()
  for i in "${a1[@]}"; do
    for j in "${a2[@]}"; do
      if [[ $i -eq $j ]]; then
        common+=($i)
      fi
    done
  done

  res=(${common[@]})
}


# given a PG, find another one with a disjoint active set
# 1 - the PG
# 2 - the dictionary of active sets
# 3 - [out] - the PG with a disjoint active set
function find_disjoint_pg {
  local pg=$1
  local -n ac_dict=$2
  local -n res=$3

  for cand in "${!ac_dict[@]}"; do
    if [[ $cand != $pg ]]; then
      local -i common=0
      count_common_active $pg $cand ac_dict common
      if [[ $common -eq 0 ]]; then
        res=$cand
        return
      fi
    fi
  done
}

# echo "Testing the find_disjoint_pg function"
# rs4=""
# find_disjoint_pg "2.5" pg_ac rs4
# echo "The result: $rs4"

# given a PG, find another one with a disjoint active set
# - but allow a possible common Primary
# 1 - the PG
# 2 - the dictionary of active sets
# 3 - [out] - the PG with a disjoint active set
function find_disjoint_but_primary {
  local pg=$1
  local -n ac_dict=$2
  local -n p_dict=$3
  local -n res=$4

  for cand in "${!ac_dict[@]}"; do
    if [[ "$cand" != "$pg" ]]; then
      local -i common=0
      count_common_active "$pg" "$cand" ac_dict common
      if [[ $common -eq 0 || ( $common -eq 1 && "${p_dict[$pg]}" == "${p_dict[$cand]}" )]]; then
        res=$cand
        return
      fi
    fi
  done
}

# echo "Testing the find_disjoint_but_primary function"
# rs5=""
# find_disjoint_but_primary "2.5" pg_ac pg_pr rs5
# echo "The result: $rs5"


function wait_initial_scrubs() {
    local pg_to_prim_dict=$1
    local extr_dbg=2 # note: 3 and above leave some temp files around
    (( extr_dbg >= 1 )) && echo "waiting initial" && bin/ceph pg dump pgs --format=json-pretty | \
      jq '.pg_stats | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'

    # set a long schedule for the periodic scrubs. Wait for the
    # initial 'no previous scrub is known' scrubs to finish for all PGs.
    bin/ceph tell osd.* config set osd_scrub_min_interval 7200
    bin/ceph tell osd.* config set osd_deep_scrub_interval 14400
    bin/ceph tell osd.* config set osd_max_scrubs 32
    bin/ceph tell osd.* config set osd_scrub_sleep "3.0"

    for pg in "${!pg_to_prim_dict[@]}"; do
      echo "l. 188: <$pg>"
      bin/ceph tell $pg scrub
    done

    (( extr_dbg >= 1 )) && bin/ceph pg dump pgs --format=json-pretty | \
      jq '.pg_stats | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'

    tout=40
    while [ $tout -gt 0 ] ; do
      echo " WAIT $tout"
      sleep 0.5
      #bin/ceph pg dump pgs --format=json-pretty | \
      #  jq '.pg_stats | map(select(.last_scrub_duration == 0)) | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'

      # should this be 'jq -s'? check RRR
      not_done=$(bin/ceph pg dump pgs --format=json-pretty | \
        jq '.pg_stats | map(select(.last_scrub_duration == 0)) | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})' | wc -l )
      # note that we should ignore a header line
      if [ "$not_done" -le 1 ]; then
        break
      fi
      not_done=$((not_done - 1))

      echo "Still waiting for $not_done PGs to finish initial scrubs"
      tout=$(($tout - 1))
    done

    (( extr_dbg >= 1 )) && bin/ceph pg dump pgs --format=json-pretty | \
      jq '.pg_stats | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'
    (( tout == 0 )) && return 1
}


function get_asok_dir() {
    local CEPH_ASOK_DIR=$(bin/ceph-conf --lookup asok_dir)
    if [ -n "$CEPH_ASOK_DIR" ]; then
        echo "$CEPH_ASOK_DIR"
    else
        echo ${TMPDIR:-/tmp}/ceph-asok.$$
    fi
}

function get_asok_path() {
    local name=$1
    echo /home/rfriedma/src/k3/ceph/build/asok/ceph-$name.asok
#     if [ -n "$name" ]; then
#         echo $(get_asok_dir)/ceph-$name.asok
#     else
#         echo $(get_asok_dir)/\$cluster-\$name.asok
#     fi
}

function set_query_debug() {
    local pgid=$1
    local prim_osd=`bin/ceph pg dump pgs_brief | \
      awk -v pg="^$pgid" -n -e '$0 ~ pg { print(gensub(/[^0-9]*([0-9]+).*/,"\\\\1","g",$5)); }' `

    echo "Setting scrub debug data. Primary for $pgid is $prim_osd"
    get_asok_path osd.$prim_osd
    echo "Setting scrub debug data. Primary for $pgid is $prim_osd"
    CEPH_ARGS='' bin/ceph --format=json daemon $(get_asok_path osd.$prim_osd) \
          scrubdebug $pgid set sessions
}




function TEST_abort_periodic_for_operator() {
    local dir=$1
    local -A cluster_conf=(
        ['osds_num']="6" 
        ['pgs_in_pool']="16"
        ['pool_name']="test"
    )
    set -x

    #standard_scrub_wpq_cluster "$dir" cluster_conf 3 || return 1
    #local poolid=${cluster_conf['pool_id']}
    #local poolname=${cluster_conf['pool_name']}


    #modified for the Jupyter environment

   # the cluster was already created

    local poolid=$pl1_num
    local poolname="pl1"
    echo "Pool: $poolname : $poolid"

    set +x
    # fill the pool with some data
    local TESTDATA="/tmp/testdata.$$"
    dd if=/dev/urandom of="$TESTDATA" bs=1032 count=1
    for i in $( seq 1 25 )
    do
        bin/rados -p "$poolname" put "obj${i}" "$TESTDATA" 2>&1 1>/dev/null
    done
    rm -f "$TESTDATA"


    # create the dictionary of the PGs in the pool
    declare -A pg_pr
    declare -A pg_ac
    declare -A pg_po
    build_pg_dicts "$dir" pg_pr pg_ac pg_po "-"

    echo "PGs data:"
    for pg in "${!pg_pr[@]}"; do
      echo "Got: $pg: ${pg_pr[$pg]} ( ${pg_ac[$pg]} ) ${pg_po[$pg]}"
    done

    for pg in "${!pg_pr[@]}"; do
      echo "bin/ceph tell $pg scrub"
      bin/ceph tell $pg scrub || return 1
    done
    set -x

    wait_initial_scrubs pg_pr

    bin/ceph tell osd.2 dump_scrub_reservations --format=json-pretty
    # limit all OSDs to one scrub at a time
    bin/ceph tell osd.* config set osd_max_scrubs 1

    # configure for slow scrubs
    bin/ceph tell osd.* config set osd_scrub_sleep 3
    bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 2
    bin/ceph tell osd.* config set osd_scrub_chunk_max 2

    # the first PG to work with:
    local pg1="1.0"
    # and another one, that shares its primary, and at least one more active set member
    local pg2=""
     for pg in "${!pg_pr[@]}"; do
      if [[ "${pg_pr[$pg]}" == "${pg_pr[$pg1]}" ]]; then
        local -i common=0
        count_common_active $pg $pg1 pg_ac common
        if [[ $common -gt 1 ]]; then
          pg2=$pg
          break
        fi
      fi
    done
    if [[ -z "$pg2" ]]; then
      # \todo handle the case when no such PG is found
      echo "No PG found with the same primary as $pg1"
      return 1
    fi

    echo "The primary (${pg_pr[$pg1]}) is allowed two concurrent scrubs"
    bin/ceph tell osd."${pg_pr[$pg1]}" config set osd_max_scrubs 2
    echo "=xxx==================== $pg1 ================== $pg2 =============================="
    # collect the timestamps before issuing the scrub command
    set_query_debug "$pg1"
    echo "<<1>>>: query:"
    echo
    bin/ceph pg "$pg1" query
    before_stamp=$(bin/ceph pg "$pg1" query | jq '.info.stats.last_deep_scrub_stamp')
    bin/ceph tell $pg1 schedule-deep-scrub

    sleep 1
    echo ' after 1 second'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber.active'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber.is_reserving_replicas'

    for i in $( seq 1 10 )
    do
      sleep 0.5
      stt=$(bin/ceph pg "$pg1" query | jq '.scrubber')
      is_active=$(echo $stt | jq '.active')
      is_reserving_replicas=$(echo $stt | jq '.is_reserving_replicas')
      if [[ "$is_active" = "true" && "$is_reserving_replicas" = "false" ]]; then
          break
      fi
      echo "Still waiting: $stt"
    done
    if [[ "$is_active" != "true" || "$is_reserving_replicas" != "false" ]]; then
      echo "The scrub is not active or is reserving replicas"
      return 1
    fi

    #sleep 1
    # make sure the scrub is in progress (past the registration stage)
    #while [ $(bin/ceph pg "$pg1" query | f json-pretty | jq '.scrubber.'  ) -gt 0 ] ; do

    #echo ' after 4 second'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber'
    #bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber.active'
    #bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber.is_reserving_replicas'


    echo "===================================================================================="

    bin/ceph tell osd.2 dump_scrub_reservations --format=json-pretty

    # now - the 2'nd scrub - which should be blocked on reserving
    set_query_debug "$pg2"
    bin/ceph tell "$pg2" schedule-deep-scrub
    sleep 0.5
    bin/ceph tell 1.6 schedule-deep-scrub
    bin/ceph tell 1.1 schedule-deep-scrub

    bin/ceph tell osd.2 dump_scrub_reservations --format=json-pretty
    bin/ceph tell osd.1 dump_scrub_reservations --format=json-pretty

    echo
    echo "<<2>>>: query:"
    echo
    #bin/ceph pg "$pg2" query
    echo "===================================================================================="
    bin/ceph pg "$pg2" query -f json-pretty | jq '.scrubber'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber'
    sleep 2
    echo "===================================================================================="
    bin/ceph pg "$pg2" query -f json-pretty | jq '.scrubber'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber'

    # make sure pg2 scrub is stuck in the reserving state
    local stt2=$(bin/ceph pg "$pg2" query | jq '.scrubber')
    local pg2_is_reserving
    pg2_is_reserving=$(echo $stt2 | jq '.is_reserving_replicas')
    if [[ "$pg2_is_reserving" != "true" ]]; then
      echo "The scheduled scrub for $pg2 should have been stuck"
      return 1
    fi

    # now - issue an operator-initiated scrub on pg2.
    # The periodic scrub should be aborted, and the operator-initiated scrub should start.
    bin/ceph tell "$pg2" scrub
    for i in $( seq 1 10 )
    do
      sleep 0.5
      stt2=$(bin/ceph pg "$pg2" query | jq '.scrubber')
      pg2_is_active=$(echo $stt2 | jq '.active')
      pg2_is_reserving=$(echo $stt2 | jq '.is_reserving_replicas')
      if [[ "$pg2_is_active" = "true" && "$pg2_is_reserving" != "true" ]]; then
            break
      fi
      echo "Still waiting: $stt2"
    done

    if [[ "$pg2_is_active" != "true" || "$pg2_is_reserving" = "true" ]]; then
      echo "The high-priority scrub for $pg2 is not active or is reserving replicas"
      return 1
    fi

#     do
#       sleep 0.5
#       echo "PG1 
#       after_stamp=$(bin/ceph pg "$pg1" query | jq '.info.stats.last_deep_scrub_stamp')
#       echo "Before: $before_stamp, After: $after_stamp"
#       [[ "$before_stamp" != "$after_stamp" ]] && break
#       # \todo add a timeout
#     done


#     # find two disjoint PGs
#     local pg1="1.0"
#     local pg2=""
#     find_disjoint_pg "$pg1" pg_ac pg2
#     echo "Totally disjoint: $pg1 and $pg2"
# 
#     local pg3="1.e"
#     local pg4=""
#     find_disjoint_but_primary "$pg3" pg_ac pg_pr pg4
#     echo "Same primary allowed: $pg3 and $pg4"
}

TEST_abort_periodic_for_operator "."


echo "done"

In [ ]:
%%bash

# finding out that all PGs were scrubbed at least once
bin/ceph pg dump pgs --format=json-pretty | jq '.pg_stats | map(select(.last_scrub_duration != "0")) | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'
bin/ceph pg dump pgs --format=json-pretty | jq '.pg_stats | map(select(.last_scrub_duration == "0")) | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})' | wc -l


In [ ]:
raise SystemExit("Stop here")

# Termination


In [ ]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
cd $CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    cd $jp
    %env CEPH_JTEST_ROOT=$jp
fi

pwd

../src/stop.sh
sleep 4
../src/stop.sh
